# A02. AI가입 전/후 이체 유형 비교

- 입력: 고객정보, AI요청로그, AI이체실행로그
- 노출기간(가입 경과일) 고려한 지표와 퍼널 확인

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

root = Path.cwd()
refactor_dir = root if root.name == 'refactor' else (root / 'analysis' / 'refactor')
if not refactor_dir.exists():
    refactor_dir = Path('/workspace/analysis/refactor')
if str(refactor_dir) not in sys.path:
    sys.path.insert(0, str(refactor_dir))

import common
import a02_transfer_prepost as a02
print('refactor_dir =', refactor_dir)

In [ ]:
PROFILE_FILE = '/home/jovyan/cjs/refactor2/data/user.csv'
CHAT_FILE = '/home/jovyan/cjs/refactor2/data/chat.csv'
AI_TRANSFER_FILE = '/home/jovyan/cjs/refactor2/data/ai_transfer.csv'

profile = common.normalize_profile(common.load_csv(PROFILE_FILE))
chat = common.normalize_chat(common.load_csv(CHAT_FILE))
ai_transfer = common.normalize_ai_transfer(common.load_csv(AI_TRANSFER_FILE))

print('profile:', profile.shape)
print('chat:', chat.shape)
print('ai_transfer:', ai_transfer.shape)

In [ ]:
asof_date = chat['chat_date'].max() if 'chat_date' in chat.columns and not chat.empty else pd.Timestamp.today()

metrics = a02.build_transfer_window_metrics(profile)
exposure = a02.age_exposure_table(profile, asof_date)
funnel = a02.build_funnel(profile, chat, ai_transfer)
print(a02.interpretation(metrics, exposure, funnel))

display(metrics)
display(exposure)
display(funnel)

fig, axes = plt.subplots(1,2, figsize=(14,4))
sns.barplot(data=funnel, x='stage', y='ratio_vs_signup', color='#4c72b0', ax=axes[0])
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_title('AI가입자 대비 퍼널 전환률')

plot_m = metrics[metrics['metric'].str.contains('평균', na=False)].copy()
sns.barplot(data=plot_m, y='metric', x='value', palette='viridis', ax=axes[1])
axes[1].set_title('이체 평균 지표')
plt.tight_layout(); plt.show()